<a href="https://colab.research.google.com/github/LiNlIn4968/cifar10/blob/main/CIM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ==========================================
# 1. 标准 BNN 直通估计器 (STE)
# ==========================================
class Binarize(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return x.sign()

    @staticmethod
    def backward(ctx, grad_output):
        x, = ctx.saved_tensors
        grad_input = grad_output.clone()
        # 梯度截断，防止权重跑飞
        grad_input[x.abs() > 1.0] = 0
        return grad_input

def binarize(x):
    return Binarize.apply(x)

# ==========================================
# 2. 完美适配 1x64 阵列的 XNOR-Net 架构
# ==========================================
class CIM_MNIST_Net(nn.Module):
    def __init__(self):
        super().__init__()
        # 隐藏层设为 64，意味着我们需要 64 条计算列，但如果你只有 1x64，
        # 在硬件实现上可以通过多次复用来计算，这里算法上保持 64 保证准确率
        self.fc1 = nn.Linear(784, 64, bias=False)
        self.bn1 = nn.BatchNorm1d(64) # 极其重要：保留 BN 层
        self.fc2 = nn.Linear(64, 10, bias=False)

    def forward(self, x):
        x = x.view(-1, 784)
        # 【修正核心 1】严格的输入双极性化 (-1, +1)
        x = (x > 0.0).float() * 2 - 1

        # 【修正核心 2】在计算点积之前，先对权重进行二值化
        w1_bin = binarize(self.fc1.weight)

        # 执行点积 (这才是真正的 XNOR 逻辑训练)
        n = F.linear(x, w1_bin)

        x = self.bn1(n)
        x = self.fc2(x)
        return x

# ==========================================
# 3. 数据与训练 (5 Epochs 就能飙升)
# ==========================================
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_loader = DataLoader(datasets.MNIST('./data', train=True, download=True, transform=transform), batch_size=128, shuffle=True)
test_loader = DataLoader(datasets.MNIST('./data', train=False, transform=transform), batch_size=1000, shuffle=False)

model = CIM_MNIST_Net()
# 降低学习率，让模型稳稳地收敛
optimizer = optim.Adam(model.parameters(), lr=0.002)

print("Step 1: Starting XNOR Training (Expected to reach ~90%+)...")
for epoch in range(5):
    model.train()
    for data, target in train_loader:
        optimizer.zero_grad()
        output = model(data)
        loss = nn.CrossEntropyLoss()(output, target)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1} finished.")

# ==========================================
# 4. 硬件感知评估 (1x64 阵列时分复用模型)
# ==========================================
def hardware_aware_eval(model, loader, v_step=3.58, noise_sigma=5.0):
    model.eval()
    correct = 0
    with torch.no_grad():
        w_bin = model.fc1.weight.sign()
        for data, target in loader:
            x = (data.view(-1, 784) > 0.0).float() * 2 - 1

            # 记录数字域累加的总结果
            n_hardware_total = torch.zeros(x.size(0), 64).to(x.device)

            # 【高级模拟】将 784 个输入切分为 13 次 1x64 阵列的计算！
            for i in range(0, 784, 64):
                x_chunk = x[:, i:i+64]
                w_chunk = w_bin[:, i:i+64]

                # 1. 硬件单元的点积 (最大范围 -64 到 64)
                n_chunk = torch.matmul(x_chunk, w_chunk.t())

                # 2. 模拟域电压与噪声
                v_mbl = 400.0 + (n_chunk * v_step)
                v_noisy = v_mbl + torch.randn_like(v_mbl) * noise_sigma

                # 3. 4-bit ADC 量化
                # 设定 1x64 阵列的极限摆幅为 400 +/- (64 * 3.58) = 400 +/- 229 mV
                v_min, v_max = 170.0, 630.0
                v_q = torch.clamp(v_noisy, v_min, v_max)

                # 划分 16 个等级 (0-15)
                adc_level = torch.round((v_q - v_min) / ((v_max - v_min) / 15))

                # 4. 数字域重构
                # Level 7.5 代表 0 电压偏移。还原回点积数值供下一层计算
                adc_centered = adc_level - 7.5
                n_reconstructed = adc_centered * ( ((v_max - v_min) / 15) / v_step )

                # 5. 数字累加器累加 13 次结果
                n_hardware_total += n_reconstructed

            # 【关键】将硬件重构的数值送入 BN 层进行归一化，才能匹配下一层权重
            x_bn = model.bn1(n_hardware_total)
            output = model.fc2(x_bn)

            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

    return correct / len(loader.dataset)

# ==========================================
# 5. 运行结果输出
# ==========================================
print("\nStep 2: Hardware Evaluation Results")
# 这里评估的是将 784 分成 13 个 64 长度的块，每次进入你 1x64 阵列的情况
acc_ideal = hardware_aware_eval(model, test_loader, v_step=3.58, noise_sigma=0.0)
print(f"Accuracy (Ideal Hardware, 0mV Noise): {acc_ideal*100:.2f}%")

acc_real = hardware_aware_eval(model, test_loader, v_step=3.58, noise_sigma=5.0)
print(f"Accuracy (With 5mV Noise): {acc_real*100:.2f}%")

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.50MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 134kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.08MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.99MB/s]


Step 1: Starting XNOR Training (Expected to reach ~90%+)...
Epoch 1 finished.
Epoch 2 finished.
Epoch 3 finished.
Epoch 4 finished.
Epoch 5 finished.

Step 2: Hardware Evaluation Results
Accuracy (Ideal Hardware, 0mV Noise): 90.77%
Accuracy (With 5mV Noise): 90.89%
